In [1]:
import yfinance as yf
import pandas as pd
from tickers import tickers_atuais

In [2]:
tickers = tickers_atuais()

In [38]:
dfx = pd.read_csv('ohlcv5y.csv')
df_ohlcv = pd.DataFrame(dfx)

dfy = pd.read_csv('fund.csv')
df_fund = pd.DataFrame(dfy)

In [ ]:
import numpy as np

modelo = pd.merge(df_ohlcv, df_fund, on="Ticker", how="left")
modelo = modelo.sort_values(['Ticker', 'Date']).reset_index(drop=True)

def add_features(g):
    # reset_index garante que o rolling/shift não vaza entre tickers
    g = g.copy()

    # ── TENDÊNCIA ──────────────────────────────────────────
    g['sma_20']         = g['Close'].rolling(20).mean()
    g['sma_50']         = g['Close'].rolling(50).mean()
    g['sma_100']        = g['Close'].rolling(100).mean()
    g['ema_12']         = g['Close'].ewm(span=12, adjust=False).mean()
    g['ema_26']         = g['Close'].ewm(span=26, adjust=False).mean()
    g['price_vs_sma20'] = g['Close'] / g['sma_20'] - 1
    g['price_vs_sma50'] = g['Close'] / g['sma_50'] - 1
    g['sma20_vs_sma50'] = g['sma_20'] / g['sma_50'] - 1

    # ── MOMENTUM / RETORNOS ────────────────────────────────
    g['return_1d']  = g['Close'].pct_change(1)
    g['return_5d']  = g['Close'].pct_change(5)
    g['return_20d'] = g['Close'].pct_change(20)
    g['return_60d'] = g['Close'].pct_change(60)

    # ── VOLATILIDADE ───────────────────────────────────────
    g['volatility_10d'] = g['return_1d'].rolling(10).std()
    g['volatility_20d'] = g['return_1d'].rolling(20).std()
    # .values evita desalinhamento de índice no pd.concat
    atr_hl = (g['High'] - g['Low']).values
    atr_hp = (g['High'].values - g['Close'].shift(1).values)
    atr_lp = (g['Low'].values  - g['Close'].shift(1).values)
    true_range = np.maximum(atr_hl, np.maximum(np.abs(atr_hp), np.abs(atr_lp)))
    g['ATR_14'] = pd.Series(true_range, index=g.index).rolling(14).mean()

    # ── OSCILADORES ────────────────────────────────────────
    delta = g['Close'].diff()
    gain  = delta.where(delta > 0, 0).rolling(14).mean()
    loss  = (-delta.where(delta < 0, 0)).rolling(14).mean()
    g['RSI'] = 100 - (100 / (1 + gain / loss.replace(0, np.nan)))

    g['MACD']        = g['ema_12'] - g['ema_26']
    g['MACD_signal'] = g['MACD'].ewm(span=9, adjust=False).mean()
    g['MACD_hist']   = g['MACD'] - g['MACD_signal']

    std_20           = g['Close'].rolling(20).std()
    g['BB_upper']    = g['sma_20'] + 2 * std_20
    g['BB_lower']    = g['sma_20'] - 2 * std_20
    band             = (g['BB_upper'] - g['BB_lower']).replace(0, np.nan)
    g['BB_position'] = (g['Close'] - g['BB_lower']) / band
    g['BB_width']    = band / g['sma_20']

    # ── VOLUME ─────────────────────────────────────────────
    g['volume_sma20'] = g['Volume'].rolling(20).mean()
    g['volume_ratio'] = g['Volume'] / g['volume_sma20']
    g['OBV']          = (np.sign(g['Close'].diff()) * g['Volume']).cumsum()

    return g

# Cada ticker é processado isoladamente — sem vazamento entre ações
modelo = modelo.groupby('Ticker', group_keys=False).apply(add_features)

# TARGET por ticker (shift -1 dentro de cada grupo)
modelo['target'] = (
    modelo.groupby('Ticker')['Close'].shift(-1) > modelo['Close']
).astype(int)

modelo = modelo.dropna()

# Validação: checar se os indicadores estão corretos por ticker
print(f"Linhas totais: {len(modelo):,} | Tickers: {modelo['Ticker'].nunique()}")
print(f"\nVerificação de isolamento (sma_20 de 3 tickers diferentes):")
for t in modelo['Ticker'].unique()[:3]:
    sub = modelo[modelo['Ticker'] == t]
    calculado = sub['Close'].rolling(20).mean().iloc[-1]
    salvo     = sub['sma_20'].iloc[-1]
    ok = "✅" if abs(calculado - salvo) < 1e-6 else "❌ ERRO"
    print(f"  {t}: sma_20 salvo={salvo:.4f}  recalculado={calculado:.4f}  {ok}")

modelo.head()


/var/folders/b_/p2n22yvj7wj8k1nq7fgyww940000gn/T/ipykernel_2888/231211063.py:21: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  g['return_1d']  = g['Close'].pct_change(1)
/var/folders/b_/p2n22yvj7wj8k1nq7fgyww940000gn/T/ipykernel_2888/231211063.py:22: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  g['return_5d']  = g['Close'].pct_change(5)
/var/folders/b_/p2n22yvj7wj8k1nq7fgyww940000gn/T/ipykernel_2888/231211063.py:23: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pc

Shape: (0, 53)
Tickers: 0 | Amostras: 0
Distribuição target: {}


/var/folders/b_/p2n22yvj7wj8k1nq7fgyww940000gn/T/ipykernel_2888/231211063.py:60: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  modelo = modelo.groupby('Ticker', group_keys=False).apply(add_features)


,Date,Ticker,Open,High,Low,Close,Volume,Setor,Indústria,Preço Atual,...,MACD_signal,MACD_hist,BB_upper,BB_lower,BB_position,BB_width,volume_sma20,volume_ratio,OBV,target


In [40]:
modelo.tail()

,Date,Ticker,Open,High,Low,Close,Volume,Setor,Indústria,Preço Atual,...,MACD_signal,MACD_hist,BB_upper,BB_lower,BB_position,BB_width,volume_sma20,volume_ratio,OBV,target


In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# criar features
features = modelo.columns[10:]

X = modelo[features]
y = modelo['Sinal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# 4. Treinar XGBoost
bst = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42
)

bst.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)


ImportError: cannot import name 'create_features_xgboost' from 'features' (/Users/lucasmazagao/projetoyahoo/features.py)

In [ ]:
preds = bst.predict(X_test)
print(f"\nAcurácia: {accuracy_score(y_test, preds):.4f}")
print("\nRelatório completo:")
print(classification_report(y_test, preds, target_names=['Desce (0)', 'Sobe (1)']))

# 6. Feature importance (top 10)
import pandas as pd
fi = pd.Series(bst.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nTop 10 features mais importantes:")
print(fi.head(10).to_string())

vamos criar entao as features de avaliacao estudando os modelos

ideias geradas para fazer

In [ ]:
# ============================================================
# PREPARAÇÃO DOS DADOS PARA OS MODELOS
# ============================================================

# 1. Ordenar por Ticker e Date (CRÍTICO para séries temporais)
df = df_historico.sort_values(['Ticker', 'Date']).reset_index(drop=True)

# 2. Criar features por ticker (indicadores técnicos)
def add_features(group):
    # Retornos
    group['return_1d'] = group['Close'].pct_change()
    group['return_5d'] = group['Close'].pct_change(5)
    
    # Médias móveis
    group['sma_5'] = group['Close'].rolling(5).mean()
    group['sma_20'] = group['Close'].rolling(20).mean()
    
    # Volatilidade (desvio padrão de 20 dias)
    group['volatility'] = group['return_1d'].rolling(20).std()
    
    # Volume relativo
    group['volume_ratio'] = group['Volume'] / group['Volume'].rolling(20).mean()
    
    # RSI simplificado (14 períodos)
    delta = group['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    group['RSI'] = 100 - (100 / (1 + gain / loss))
    
    # MACD simplificado
    ema_12 = group['Close'].ewm(span=12).mean()
    ema_26 = group['Close'].ewm(span=26).mean()
    group['MACD'] = ema_12 - ema_26
    
    # Bollinger Bands
    group['BB_upper'] = group['sma_20'] + 2 * group['Close'].rolling(20).std()
    group['BB_lower'] = group['sma_20'] - 2 * group['Close'].rolling(20).std()
    group['BB_position'] = (group['Close'] - group['BB_lower']) / (group['BB_upper'] - group['BB_lower'])
    
    return group

df = df.groupby('Ticker', group_keys=False).apply(add_features)

# 3. Criar TARGET (o que queremos prever)
# Target: 1 se o preço SOBE no próximo dia, 0 se DESCE
df['target'] = (df.groupby('Ticker')['Close'].shift(-1) > df['Close']).astype(int)

# 4. Remover linhas com NaN (início de cada série por causa dos indicadores)
df = df.dropna()

print(f"Shape final: {df.shape}")
print(f"Colunas: {df.columns.tolist()}")
df.head(10)

In [ ]:
# ============================================================
# VERIFICAR SE ESTÁ PRONTO PARA CADA MODELO
# ============================================================

# Features que vamos usar
feature_cols = ['return_1d', 'return_5d', 'sma_5', 'sma_20', 'volatility', 
                'volume_ratio', 'RSI', 'MACD', 'BB_position']

print("=" * 60)
print("XGBOOST - Pronto para usar!")
print("=" * 60)
X_xgb = df[feature_cols]
y_xgb = df['target']
print(f"X shape: {X_xgb.shape}  (amostras × features)")
print(f"y shape: {y_xgb.shape}  (targets)")
print(f"Distribuição target: {y_xgb.value_counts().to_dict()}")

print("\n" + "=" * 60)
print("LSTM - Precisa converter para 3D")
print("=" * 60)

import numpy as np
from sklearn.preprocessing import StandardScaler

SEQ_LEN = 20  # usando 20 porque temos só 1 mês de dados de teste

X_lstm, y_lstm = [], []
scaler = StandardScaler()

for ticker in df['Ticker'].unique():
    ticker_df = df[df['Ticker'] == ticker].sort_values('Date')
    
    # Normalizar features (CRÍTICO para LSTM)
    features_scaled = scaler.fit_transform(ticker_df[feature_cols])
    targets = ticker_df['target'].values
    
    # Criar sequências
    for i in range(SEQ_LEN, len(features_scaled)):
        X_lstm.append(features_scaled[i-SEQ_LEN:i])  # últimos N dias
        y_lstm.append(targets[i])

X_lstm = np.array(X_lstm)
y_lstm = np.array(y_lstm)

print(f"X shape: {X_lstm.shape}  (amostras × sequência × features)")
print(f"y shape: {y_lstm.shape}  (targets)")

print("\n" + "=" * 60)
print("GARCH - Usa só retornos de cada ticker")
print("=" * 60)
print("Para GARCH, filtrar por ticker e usar coluna 'return_1d'")
print(f"Exemplo NVDA: {len(df[df['Ticker'] == df['Ticker'].iloc[0]])} observações")